# Understanding Joo Kai's DDQN-Based MTD System (MTDShield)

This notebook walks through Joo Kai Tay's thesis *"Using Artificial Intelligence to Automate the Deployment of Moving Target Defence Operations"* (GENG5512, UWA, Oct 2024). It explains the technical gap, the reinforcement learning framework, the DDQN algorithm, reward structure, assumptions, limitations, and how the trained model works — all explained for a CS student without a heavy math background.

---

## 1. The Story: What Problem Was Joo Kai Trying to Solve?

### The Narrative Arc (Introduction & Literature Review)

Joo Kai's thesis tells this story:

1. **Static defences are insufficient** — Firewalls, IDS, antivirus etc. maintain a fixed configuration. Attackers can observe, learn, and eventually exploit them.

2. **MTD is the answer** — Moving Target Defence continuously changes the attack surface (IPs, OS, services, topology) so attackers can't rely on stale reconnaissance. Even if they find a vulnerability, it may no longer be accessible.

3. **But existing MTD research has a gap** — Most MTD research focuses on:
   - Optimising *specific* MTD techniques for *specific* attackers
   - Time-based deployment (fixed intervals) with no regard for the network's actual security state
   - Individual techniques rather than *combinations*

4. **The technical gap he identified:**
   > *"Current approaches towards MTD predominantly focus on optimising specific MTD techniques tailored to specific types of attackers... This approach can inadvertently limit the understanding of MTD's full capabilities and its effectiveness against diverse or evolving cyber threats."*

   In plain terms: **Nobody was building a system that could look at the network's current security posture and intelligently decide WHICH MTD technique to deploy and WHEN, across multiple strategies simultaneously.**

5. **His proposed solution (MTDShield):** Use reinforcement learning (specifically DDQN) to create an agent that:
   - Observes network security metrics in real-time
   - Decides whether to deploy an MTD operation (or do nothing)
   - Selects which specific MTD technique to use
   - Learns from the consequences of its decisions

### Technical Gaps From Literature Review

| Gap | Source | How Joo Kai Addresses It |
|-----|--------|------------------------|
| MTD combinations not well-studied for both attack AND defence metrics | Alavizadeh et al. — only looked at attack metrics | Uses 5 metrics spanning both attack (ASR, ROA, Risk) and defence (MTTC, APE) |
| Time-based MTD is predictable; attackers can adapt | Winterrose et al. — showed adaptive attackers exploit time patterns | RL agent deploys reactively based on network state, not fixed schedule |
| Existing RL approaches have scalability issues | Zhang et al. (EVADE) — DRL overhead in SDN controller | Uses dual-pathway neural network with LSTM for temporal awareness |
| No temporal awareness in MTD decisions | Kreischer — FL+RL but ignored temporal aspects | LSTM module processes time-series security metrics |
| Static security models don't capture dynamic networks | Yusuf et al. (T-HARM) | Uses real-time metrics from running simulation |

## 2. The Reinforcement Learning Framework

### Quick RL Primer

Reinforcement Learning has four key components:

```
┌─────────────────────────────────────────────────────────┐
│                    THE RL LOOP                          │
│                                                         │
│   ┌───────────┐    action     ┌──────────────────┐     │
│   │           │──────────────>│                  │     │
│   │   AGENT   │               │   ENVIRONMENT    │     │
│   │ (MTDShield│<──────────────│   (Network +     │     │
│   │  DDQN)    │  state,reward │    Attacker)     │     │
│   └───────────┘               └──────────────────┘     │
│                                                         │
│   The agent observes the environment's state,           │
│   takes an action, receives a reward, and learns.       │
└─────────────────────────────────────────────────────────┘
```

### In Joo Kai's System Specifically:

| RL Component | What It Is In MTDShield |
|---|---|
| **Agent** | The DDQN neural network (MTDShield) — it decides which MTD to deploy |
| **Environment** | The SimPy network simulation with attacker running continuously |
| **State** | 11 security metrics extracted from the network (5 static + 6 time-series) |
| **Actions** | 5 discrete choices: do nothing, IP Shuffle, OS Diversity, Service Diversity, Complete Topology Shuffle |
| **Reward** | Weighted sum of changes in security metrics after an action (details in Section 5) |
| **Episode** | One full simulation run — starts with fresh network, ends when network is compromised |

## 3. The Markov Decision Process (MDP)

### What Is an MDP?

An MDP is a mathematical framework for decision-making where:
- The outcome depends *only* on the current state and action (not on history — the "Markov property")
- The environment is stochastic (random elements)

Think of it like a board game: you look at the board (state), make a move (action), the game updates (transition), and you get points or penalties (reward).

### The MDP in MTDShield

```
MDP = (S, A, T, R, γ)

S = State Space     → 11 security metrics (see below)
A = Action Space    → {do_nothing, IP_Shuffle, OS_Diversity, Service_Diversity, Topology_Shuffle}
T = Transitions     → Determined by the simulation (attacker progresses, MTD effects take hold)
R = Reward function → Weighted deltas in security metrics
γ = Discount factor → 0.95 (how much to value future rewards vs immediate ones)
```

### How Decisions Happen (Step by Step)

```
Time ──────────────────────────────────────────────────────────────────>

  t=0          t=200         t=400         t=600         t=800
  │             │             │             │             │
  ▼             ▼             ▼             ▼             ▼
  Observe    Observe       Observe       Observe       Network
  state₀     state₁        state₂        state₃        Compromised!
  │             │             │             │             │
  Choose     Choose        Choose        Choose         END
  action₀    action₁       action₂       action₃       (episode)
  (Topology) (Nothing)     (IP Shuffle)  (OS Div)       │
  │             │             │             │             │
  Get        Get           Get           Get            │
  reward₀    reward₁       reward₂       reward₃       │
```

### Important: The Agent Does NOT Default to Any Action

The agent doesn't "default to shuffle" with other operations as random chances. Instead:

1. **During training (high epsilon):** The agent takes mostly *random* actions to explore
2. **As training progresses (epsilon decays):** The agent increasingly picks the action with the highest predicted Q-value
3. **After training:** The agent almost always picks the best learned action (epsilon ≈ 0.01)

The only forced action is the **static degradation factor**: if no MTD has been deployed for >2000 time units, a random MTD is forced. This is a safety net, not part of the learned policy.

### The Trigger Interval

The agent makes decisions at exponentially-distributed intervals (mean = `mtd_interval`, typically 200 time units). This means:
- On average, the agent considers acting every ~200 time units
- But the actual interval is random (exponential distribution) — sometimes sooner, sometimes later
- At each decision point, the agent can choose to do nothing (action 0) or deploy one of the 4 MTD techniques

## 4. The State Space: What the Agent Sees

The agent observes 11 features, split into two groups:

### Static Features (5) — Current Snapshot of Network Health

| Feature | What It Measures | Good Direction |
|---------|-----------------|----------------|
| `host_compromise_ratio` | Fraction of hosts compromised (in last 60 time units) | ↓ Lower is better |
| `attack_path_exposure` | How many attack paths exist to critical assets | ↓ Lower is better |
| `overall_asr_avg` | Attack Success Rate — how often the attacker succeeds | ↓ Lower is better |
| `roa` | Return on Attack — attacker's "profit" from exploiting vulns | ↓ Lower is better |
| `risk` | Cumulative vulnerability risk score | ↓ Lower is better |

### Time-Series Features (6) — Temporal/Dynamic Metrics

| Feature | What It Measures | Good Direction |
|---------|-----------------|----------------|
| `mtd_freq` | How often MTDs have been deployed | Informational |
| `overall_mttc_avg` | Mean Time to Compromise — how long attacks take | ↑ Higher is better |
| `time_since_last_mtd` | Time elapsed since last MTD deployment | Informational |
| `shortest_path_variability` | How much the shortest attack paths change | ↑ Higher is better |
| `ip_variability` | How much IP addresses have changed | ↑ Higher is better |
| `attack_type` | What the attacker is currently doing (1-6 encoded) | Informational |

In [ ]:
# Visualise: The state features and their roles
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('MTDShield State Space Architecture', fontsize=16, fontweight='bold', pad=20)

# Static features box
static_box = mpatches.FancyBboxPatch((0.5, 5.5), 5.5, 4, boxstyle="round,pad=0.3", 
                                       facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax.add_patch(static_box)
ax.text(3.25, 9.1, 'STATIC FEATURES (5)', ha='center', fontsize=12, fontweight='bold', color='#1565C0')
static_features = [
    ('host_compromise_ratio', '↓ minimize'),
    ('attack_path_exposure', '↓ minimize'),
    ('overall_asr_avg', '↓ minimize'),
    ('roa (return on attack)', '↓ minimize'),
    ('risk', '↓ minimize'),
]
for i, (feat, direction) in enumerate(static_features):
    ax.text(1.0, 8.5 - i*0.6, f'• {feat}', fontsize=10, family='monospace')
    ax.text(5.0, 8.5 - i*0.6, direction, fontsize=9, color='#C62828')

# Time-series features box
ts_box = mpatches.FancyBboxPatch((0.5, 0.3), 5.5, 4.8, boxstyle="round,pad=0.3",
                                   facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax.add_patch(ts_box)
ax.text(3.25, 4.7, 'TIME-SERIES FEATURES (6)', ha='center', fontsize=12, fontweight='bold', color='#2E7D32')
ts_features = [
    ('mtd_freq', 'informational'),
    ('overall_mttc_avg', '↑ maximize'),
    ('time_since_last_mtd', 'informational'),
    ('shortest_path_variability', '↑ maximize'),
    ('ip_variability', '↑ maximize'),
    ('attack_type (1-6)', 'informational'),
]
for i, (feat, direction) in enumerate(ts_features):
    ax.text(1.0, 4.1 - i*0.6, f'• {feat}', fontsize=10, family='monospace')
    color = '#2E7D32' if 'maximize' in direction else '#757575'
    ax.text(5.0, 4.1 - i*0.6, direction, fontsize=9, color=color)

# Neural network box
nn_box = mpatches.FancyBboxPatch((7.5, 2.5), 3, 5, boxstyle="round,pad=0.3",
                                   facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax.add_patch(nn_box)
ax.text(9.0, 7.1, 'DDQN NETWORK', ha='center', fontsize=12, fontweight='bold', color='#E65100')
ax.text(9.0, 6.3, 'Dense(128)→Dense(64)', ha='center', fontsize=9, family='monospace')
ax.text(9.0, 5.7, 'LSTM(64)→LSTM(32)', ha='center', fontsize=9, family='monospace')
ax.text(9.0, 5.1, 'Concatenate + Fuse', ha='center', fontsize=9, family='monospace')
ax.text(9.0, 4.5, '→ Dense(64)', ha='center', fontsize=9, family='monospace')
ax.text(9.0, 3.8, '→ Dense(5) = Q-values', ha='center', fontsize=9, family='monospace', fontweight='bold')

# Action box
action_box = mpatches.FancyBboxPatch((11.5, 2.5), 2.2, 5, boxstyle="round,pad=0.3",
                                      facecolor='#FCE4EC', edgecolor='#AD1457', linewidth=2)
ax.add_patch(action_box)
ax.text(12.6, 7.1, 'ACTIONS', ha='center', fontsize=12, fontweight='bold', color='#AD1457')
actions = ['0: Do Nothing', '1: Topology\n   Shuffle', '2: IP Shuffle', '3: OS Diversity', '4: Service\n   Diversity']
for i, act in enumerate(actions):
    ax.text(11.8, 6.3 - i*0.8, act, fontsize=9, family='monospace')

# Arrows
ax.annotate('', xy=(7.4, 7.0), xytext=(6.1, 7.0),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(7.4, 3.5), xytext=(6.1, 3.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.annotate('', xy=(11.4, 5.0), xytext=(10.6, 5.0),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

plt.tight_layout()
plt.show()

## 5. The DDQN Algorithm — Explained Without Heavy Math

### Step 1: What is Q-Learning?

Imagine you're playing a game. At each point, you have several moves. **Q-learning** tries to learn a "quality score" (Q-value) for every possible combination of:
- Where you are (state)
- What you could do (action)

The Q-value tells you: *"If I take this action from this state, and then play optimally from there, what total reward can I expect?"*

### Step 2: The Bellman Equation (Intuition)

The core idea is recursive:

```
Q(state, action) = immediate_reward + γ × best_future_Q_value
```

In plain English:
> *"The value of doing this action here = what I get right now + (discounted) the best I can do from the next state"*

- **γ (gamma) = 0.95** means future rewards are worth 95% of their face value per time step
- A reward 10 steps in the future is worth 0.95¹⁰ ≈ 0.60 of its value now
- This prevents the agent from valuing infinitely far-off rewards equally to immediate ones

### Step 3: Why "Deep" Q-Learning?

With 11 continuous input features, you can't store Q-values in a table (infinite states). Instead, you use a **neural network** to approximate Q(state, action) — hence "Deep" Q-Learning.

The neural network takes in the state and outputs Q-values for ALL actions simultaneously:

```
Input: [state features]  →  Neural Network  →  Output: [Q₀, Q₁, Q₂, Q₃, Q₄]
                                                         │    │    │    │    │
                                                     nothing IP  OS  Svc Topo
```

Then you pick the action with the highest Q-value.

### Step 4: The Problem with Regular DQN

Regular DQN has an **overestimation problem**. Why?

The same network that *picks* the best action also *evaluates* how good that action is. This is like asking someone to both nominate themselves for an award AND judge their own nomination — they'll be biased towards overvaluing themselves.

```
Regular DQN (overestimates):
  target = reward + γ × max[same_network.predict(next_state)]
                         ↑
                    Same network selects AND evaluates
                    → Tends to pick the noisily high value
                    → Overoptimistic Q-values
                    → Bad policies
```

### Step 5: How DDQN Fixes This

**Double DQN** uses TWO networks:
- **Main network**: Picks which action looks best
- **Target network**: Evaluates how good that action actually is

```
Double DQN (more accurate):
  best_action = argmax[main_network.predict(next_state)]     ← Main picks
  target = reward + γ × target_network.predict(next_state)[best_action]  ← Target evaluates
```

The target network is a **delayed copy** of the main network, updated every 10 episodes. Because it's slightly behind, it provides a more stable, less biased evaluation.

In [ ]:
# Pseudocode: The DDQN replay function (what the actual code does)
# This is a readable version of mtdnetwork/mtdai/mtd_ai.py:replay()

def ddqn_replay_pseudocode():
    """
    PSEUDOCODE — not executable, for understanding only.
    
    Called after each MTD execution to train the main network.
    """
    # Don't train until we have enough experiences
    if len(memory) < train_start:  # default: 1000 experiences
        return
    
    # Sample a random batch of past experiences
    minibatch = random.sample(memory, batch_size)  # default: 32
    
    for (state, time_series, action, reward, next_state, next_time_series, done) in minibatch:
        
        # Get current Q-value predictions from main network
        target = main_network.predict(state, time_series)
        
        if done:  # Episode ended (network compromised)
            # No future rewards possible — Q-value is just the reward
            target[action] = reward
        else:
            # === THIS IS THE DOUBLE DQN MAGIC ===
            
            # Step 1: MAIN network picks the best next action
            best_next_action = argmax(main_network.predict(next_state))
            
            # Step 2: TARGET network evaluates that action
            future_q = target_network.predict(next_state)[best_next_action]
            
            # Step 3: Update Q-value using Bellman equation
            target[action] = reward + gamma * future_q
            #                  │        │        │
            #           immediate    discount  best future
            #            reward      factor     value
        
        # Train main network to output these updated targets
        main_network.fit(state, time_series, target)
    
    # Decay exploration rate
    epsilon *= epsilon_decay  # e.g., 1.0 * 0.995 = 0.995 after episode 1

print("Above is pseudocode for understanding — see mtdnetwork/mtdai/mtd_ai.py for actual implementation")

### Step 6: Exploration vs. Exploitation (ε-greedy)

The agent needs to balance:
- **Exploration**: Try random actions to discover new strategies
- **Exploitation**: Use what it's already learned to pick the best action

This is controlled by **epsilon (ε)**:

```
if random() < epsilon:
    action = random_action()      # Explore: pick randomly
else:
    action = best_Q_value_action() # Exploit: pick the learned best
```

Epsilon starts at 1.0 (100% random) and decays each episode:

```
Episode  0: ε = 1.000  (100% random)
Episode 10: ε = 0.951  (95% random)
Episode 50: ε = 0.778  (78% random)
Episode100: ε = 0.606  (61% random)
Episode200: ε = 0.367  (37% random)
Episode460: ε = 0.010  (1% random — minimum reached)
```

**Joo Kai's finding:** Lower starting epsilon (0.5-0.7) actually performed better than 1.0 in his experiments. This is counterintuitive — it means for this problem, early exploitation of obvious good strategies (like deploying MTD when risk is high) outperformed extensive exploration, likely because the simulation runtime was limited.

In [ ]:
# Visualise epsilon decay over episodes
import numpy as np
import matplotlib.pyplot as plt

episodes = np.arange(500)
epsilon_min = 0.01

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Different starting epsilons with decay=0.995
for eps_start in [1.0, 0.8, 0.6, 0.5]:
    epsilon_values = [max(eps_start * (0.995 ** e), epsilon_min) for e in episodes]
    ax1.plot(episodes, epsilon_values, label=f'ε₀={eps_start}')

ax1.set_xlabel('Episode')
ax1.set_ylabel('Epsilon (exploration rate)')
ax1.set_title('Epsilon Decay: Different Starting Values\n(decay=0.995)')
ax1.legend()
ax1.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='50% exploration')
ax1.set_ylim(-0.05, 1.05)
ax1.grid(True, alpha=0.3)

# Different decay rates with epsilon=1.0
for decay in [0.998, 0.995, 0.99, 0.98]:
    epsilon_values = [max(1.0 * (decay ** e), epsilon_min) for e in episodes]
    ax2.plot(episodes, epsilon_values, label=f'decay={decay}')

ax2.set_xlabel('Episode')
ax2.set_ylabel('Epsilon (exploration rate)')
ax2.set_title('Epsilon Decay: Different Decay Rates\n(ε₀=1.0)')
ax2.legend()
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. The Neural Network Architecture

MTDShield uses a **dual-pathway architecture** — two separate processing streams that merge:

```
PATHWAY 1: Static Feature Extraction          PATHWAY 2: Time-Series Analysis
─────────────────────────────────              ─────────────────────────────────
Input(5 features)                              Input(6 features, 1 timestep)
    │                                              │
    ▼                                              ▼
Dense(128) + ReLU + BatchNorm                  LSTM(64, return_sequences) + ReLU + BatchNorm
    │                                              │
    ▼                                              ▼
Dense(64) + ReLU + BatchNorm + Dropout(0.3)    LSTM(32) + ReLU + BatchNorm + Dropout(0.3)
    │                                              │
    └──────────────┬───────────────────────────────┘
                   │
                   ▼  (Concatenate)
            FEATURE FUSION
            Dense(64) + ReLU + BatchNorm + Dropout(0.3)
                   │
                   ▼
            Q-NETWORK OUTPUT
            Dense(5)  →  [Q₀, Q₁, Q₂, Q₃, Q₄]
```

### Why This Architecture?

| Component | Purpose |
|-----------|--------|
| **Dense layers** (Pathway 1) | Extract patterns from static security metrics — "How bad is the network right now?" |
| **LSTM layers** (Pathway 2) | Capture temporal patterns — "Are things getting worse or better over time?" |
| **BatchNormalization** | Stabilises training by normalising layer inputs |
| **Dropout (30%)** | Prevents overfitting by randomly disabling neurons during training |
| **ReLU activation** | Non-linearity — lets the network learn complex patterns (outputs max(0, x)) |
| **Concatenation + Fusion** | Combines spatial and temporal understanding into a unified representation |

### A Note on the LSTM Usage

The LSTM is given input shape `(6, 1)` — 6 features, 1 timestep. This is a slight misuse of LSTM, which is designed for *sequences*. With only 1 timestep, the LSTM is essentially acting as a fancy dense layer. A true temporal approach would feed a *window* of past time-series observations (e.g., the last 10 states). This is one area for potential improvement.

In [ ]:
# The actual network creation code from mtdnetwork/mtdai/mtd_ai.py
# Reproduced here with annotations

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM, Concatenate, ReLU, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def create_network(state_size=5, action_size=5, time_series_size=6):
    """
    Creates the DDQN network with dual-pathway architecture.
    
    Parameters:
        state_size: Number of static features (default 5)
        action_size: Number of possible actions (default 5: 4 MTDs + do nothing)
        time_series_size: Number of time-series features (default 6)
    
    Returns:
        Compiled Keras model
    """
    # === PATHWAY 1: Static Feature Extraction ===
    static_input = Input(shape=(state_size,))                # Input: 5 static metrics
    x = Dense(128)(static_input)                             # 128 neurons — broad feature capture
    x = ReLU()(x)                                            # Non-linearity
    x = BatchNormalization()(x)                               # Stabilise training
    x = Dense(64)(x)                                         # 64 neurons — refine features
    x = ReLU()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)                                      # 30% dropout — prevent overfitting

    # === PATHWAY 2: Time-Series Analysis ===
    time_series_input = Input(shape=(time_series_size, 1))   # Input: 6 temporal metrics
    y = LSTM(64, return_sequences=True)(time_series_input)   # 64-unit LSTM — captures sequences
    y = ReLU()(y)
    y = BatchNormalization()(y)
    y = LSTM(32)(y)                                          # 32-unit LSTM — refines temporal patterns
    y = ReLU()(y)
    y = BatchNormalization()(y)
    y = Dropout(0.3)(y)

    # === FEATURE FUSION ===
    z = Concatenate()([x, y])                                # Merge: 64 + 32 = 96 features
    z = Dense(64)(z)                                         # Compress to 64
    z = ReLU()(z)
    z = BatchNormalization()(z)
    z = Dropout(0.3)(z)

    # === Q-NETWORK OUTPUT ===
    output = Dense(action_size)(z)                           # Output: 5 Q-values (one per action)

    model = Model(inputs=[static_input, time_series_input], outputs=output)
    model.compile(loss='mean_squared_error', optimizer=Adam(learning_rate=0.001))
    return model

# Build and display model summary
model = create_network()
model.summary()

## 7. The Reward Function — How the Agent Learns What's Good and Bad

### Core Idea

The reward is based on **how much the security metrics changed** after taking an action:

```
reward = Σ (change_in_metric × weight_for_that_metric)
```

Where `change = normalized_next_value - normalized_current_value`

### The Weight System

```
NEGATIVE WEIGHTS (we want these metrics to DECREASE):
─────────────────────────────────────────────────────
attack_path_exposure:  weight = -75    If APE goes UP → negative reward (punishment)
overall_asr_avg:       weight = -75    If ASR goes UP → negative reward (punishment)  
roa:                   weight = -75    If ROA goes UP → negative reward (punishment)
risk:                  weight = -75    If Risk goes UP → negative reward (punishment)
host_compromise_ratio: weight =  0    (Not weighted! See discussion below)

POSITIVE WEIGHTS (we want these metrics to INCREASE):
─────────────────────────────────────────────────────
overall_mttc_avg:          weight = +75   If MTTC goes UP → positive reward (good!)
shortest_path_variability: weight = +75   If paths change MORE → positive reward
ip_variability:            weight = +75   If IPs change MORE → positive reward
mtd_freq:                  weight =  0    (Not weighted)
time_since_last_mtd:       weight =  0    (Not weighted)
attack_type:               weight =  0    (Not weighted)
```

### Reward/Punishment Scenarios

| Scenario | What Happens to Metrics | Reward | Outcome |
|----------|------------------------|--------|--------|
| Deploy IP Shuffle successfully | APE↓, ip_variability↑, ASR↓ | **Positive** (good action!) | Agent learns: "IP shuffle is good when exposure is high" |
| Deploy MTD but attacker already progressed | risk↑, ROA↑ despite MTD | **Negative** (too late) | Agent learns: "Should have acted sooner" |
| Do nothing when network is safe | Metrics stable (small deltas) | **~Zero** | Agent learns: "Doing nothing when safe is fine" |
| Do nothing while attacker progresses | APE↑, ASR↑, risk↑ | **Negative** | Agent learns: "Should have deployed MTD" |
| Deploy Topology Shuffle | Paths change significantly | **Positive** (variability↑, APE↓) | Agent learns: "Topology shuffle disrupts attacks" |

### What About Terminal States?

**Yes, network compromise is the worst outcome.** When the network is fully compromised:
- The episode ends (`done = True`)
- No future rewards are possible
- The Q-value for the last action is just the immediate reward (no `+ γ × future_Q`)
- But interestingly, `done` is set to `False` in the training code — the episode *ends* but the transition isn't explicitly marked as terminal in memory. The end signal comes from the SimPy simulation stopping.

**A completely untouched network** is indeed the best state — all attack metrics stay at their initial (low) values, MTTC is high, and no hosts are compromised.

In [ ]:
# Visualise the reward structure
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Scenario 1: Good MTD deployment
ax = axes[0]
metrics = ['APE', 'ASR', 'ROA', 'Risk', 'MTTC', 'Path\nVar', 'IP\nVar']
deltas = [-0.3, -0.2, -0.1, -0.15, 0.4, 0.5, 0.6]  # normalised changes
weights = [-75, -75, -75, -75, 75, 75, 75]
contributions = [d * w for d, w in zip(deltas, weights)]
colors = ['green' if c > 0 else 'red' for c in contributions]
ax.barh(metrics, contributions, color=colors, alpha=0.7, edgecolor='black')
ax.set_xlabel('Reward Contribution')
ax.set_title(f'Good MTD Deployment\nTotal Reward: {sum(contributions):.1f}', fontweight='bold', color='green')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

# Scenario 2: MTD deployed too late
ax = axes[1]
deltas_late = [0.1, 0.2, 0.15, 0.3, -0.1, 0.1, 0.2]
contributions_late = [d * w for d, w in zip(deltas_late, weights)]
colors_late = ['green' if c > 0 else 'red' for c in contributions_late]
ax.barh(metrics, contributions_late, color=colors_late, alpha=0.7, edgecolor='black')
ax.set_xlabel('Reward Contribution')
ax.set_title(f'MTD Too Late\nTotal Reward: {sum(contributions_late):.1f}', fontweight='bold', color='red')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

# Scenario 3: Doing nothing (stable network)
ax = axes[2]
deltas_nothing = [0.02, 0.01, 0.01, 0.02, -0.01, 0.0, 0.0]
contributions_nothing = [d * w for d, w in zip(deltas_nothing, weights)]
colors_nothing = ['green' if c > 0 else ('red' if c < 0 else 'gray') for c in contributions_nothing]
ax.barh(metrics, contributions_nothing, color=colors_nothing, alpha=0.7, edgecolor='black')
ax.set_xlabel('Reward Contribution')
ax.set_title(f'Do Nothing (Stable)\nTotal Reward: {sum(contributions_nothing):.1f}', fontweight='bold', color='gray')
ax.axvline(x=0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. The Training Loop — How It All Fits Together

### High-Level Training Process

```
FOR each episode (1 to 100):
│
├── 1. Create fresh SimPy environment
├── 2. Load or create network (100 nodes, with hosts, vulns, services)
├── 3. Create attacker (Adversary)
├── 4. Start attack process (runs continuously in background)
├── 5. Start MTDAITraining (RL agent makes MTD decisions)
│
├── 6. SIMULATION RUNS until network compromised:
│   │
│   ├── Every ~200 time units (exponential):
│   │   ├── Observe state (11 metrics)
│   │   ├── Choose action (ε-greedy)
│   │   ├── Execute MTD (if action > 0)
│   │   ├── Wait for MTD to complete
│   │   ├── Observe new state
│   │   ├── Calculate reward (metric deltas × weights)
│   │   ├── Store (s, a, r, s') in memory buffer
│   │   └── Train on random batch from memory (DDQN replay)
│   │
│   └── Meanwhile: Attacker continuously scanning/exploiting
│
├── 7. Every 10 episodes: Copy main network weights to target network
├── 8. Decay epsilon (ε *= 0.995)
│
└── END: Save trained model to AI_model/main_network_{name}.h5
```

In [ ]:
# Pseudocode for the full training loop
# Based on experiments/run.py:execute_ai_training()

pseudocode = """
════════════════════════════════════════════════════════════════
FULL TRAINING LOOP (from experiments/run.py)
════════════════════════════════════════════════════════════════

# Initialise DDQN components
main_network   = create_network(state_size=5, action_size=5, time_series_size=6)
target_network = create_network(state_size=5, action_size=5, time_series_size=6)
target_network.weights = main_network.weights   # Start identical
memory = deque(maxlen=2000)                      # Experience replay buffer

epsilon = 1.0        # Start fully random
gamma = 0.95         # Discount factor
batch_size = 32      # Replay batch size
train_start = 1000   # Min experiences before training

FOR episode in range(100):
    
    # Fresh simulation each episode
    env = SimPy.Environment()
    network = load_or_create_network(nodes=100)
    attacker = Adversary(network, threshold=3)
    
    # Start concurrent processes
    attack_operation.start()        # Attacker runs continuously
    mtd_ai_training.start()         # RL agent runs concurrently
    
    # Run until network compromised
    env.run(until=network_compromised)
    
    # Update target network periodically
    IF episode % 10 == 0:
        target_network.weights = main_network.weights
    
    # Decay exploration
    epsilon = max(epsilon * 0.995, 0.01)

# Save trained model
main_network.save('AI_model/main_network.h5')
"""
print(pseudocode)

## 9. How to Run the Trained Model

### Where Are the Trained Models?

```
experiments/AI_model/
├── models_joo_kai/          # Joo Kai's trained models
│   ├── main_network_*.h5    # Various hyperparameter configurations
│   └── ...
└── models_will/             # Will's training runs
    └── ...
```

### Running Inference (Using a Trained Model)

The function `execute_ai_model()` in `experiments/run.py` loads a trained model and runs it on a fresh simulation:

```python
from experiments.run import execute_ai_model

# Run the trained model on a simulation
execute_ai_model(
    features={
        'static': ['host_compromise_ratio', 'attack_path_exposure', 'overall_asr_avg', 'roa', 'risk'],
        'time': ['mtd_freq', 'overall_mttc_avg', 'time_since_last_mtd', 
                 'shortest_path_variability', 'ip_variability', 'attack_type']
    },
    model_path='experiments/AI_model/models_joo_kai/main_network_XYZ.h5',
    total_nodes=100,
    mtd_interval=200,
    custom_strategies=[CompleteTopologyShuffle, IPShuffle, OSDiversity, ServiceDiversity],
    epsilon=0.01,           # Almost no exploration (exploit learned policy)
    new_network=True
)
```

The inference code (`MTDAIOperation`) works identically to training but:
- No replay memory / no training updates
- Low epsilon (mostly exploitation)
- Single episode (one simulation run)

## 10. Assumptions and Limitations

### Assumptions Made by Joo Kai

| Assumption | Implication |
|-----------|-------------|
| **Single attacker** | No multi-agent or coordinated attacks. Real networks face multiple simultaneous threats. |
| **Fixed attack model** | Attacker follows a fixed sequence (scan→enumerate→exploit→pivot). Real attackers are adaptive. |
| **No cost to deploying MTD** | In practice, MTD operations cause downtime, performance degradation, and monetary cost. The reward function doesn't penalise frequent deployment. |
| **Perfect MTD execution** | MTD operations always succeed. In practice, an OS diversity change might fail, or an IP shuffle might break connections. |
| **Simulation = reality** | The SimPy simulation is an abstraction. Vulnerability distributions, attack timing, and network topology are simplified. |
| **LSTM receives single timestep** | Time-series features are fed as shape (6, 1) — 6 features, 1 timestep. This doesn't leverage LSTM's sequence modelling capability. |
| **Normalisation from memory buffer** | Rewards are normalised against past experiences in the replay buffer, which shifts as training progresses. Early normalisation is unreliable. |
| **`done=False` always** | Terminal transitions (network compromise) are never flagged as `done=True`, meaning the agent doesn't explicitly learn that compromise is a terminal failure state. |
| **`host_compromise_ratio` weight = 0** | The most intuitive metric (how many hosts are compromised) has zero weight in the reward function. |

### Limitations of the Simulation Itself

| Limitation | Impact |
|-----------|--------|
| **Limited attacker model** | 6 attack actions (scan, enumerate, port scan, exploit, brute force, scan neighbor). No lateral movement sophistication, no APT persistence, no exfiltration. |
| **4 MTD operations** | Real MTD includes port randomisation, memory layout randomisation (ASLR), dynamic firewall rules, container migration, etc. |
| **No network performance modelling** | MTD operations are assumed costless. A real topology shuffle would cause significant disruption. |
| **Fixed network size** | Tested on 100-node networks. Scalability to enterprise networks (thousands of nodes) is unknown. |
| **Short simulation runtime** | Limited runtime may explain why lower epsilon (early exploitation) outperformed standard ε-greedy. |
| **No external benchmarks** | Results are compared only against the simulator's own no-MTD baseline. |

## 11. Why DDQN? And What Are the Alternatives?

### Why DDQN Is a Reasonable Fit

| Property | Why It Fits MTD |
|----------|----------------|
| **Discrete action space** | MTD has a small, finite set of actions (5). DDQN handles discrete actions naturally. |
| **Continuous state space** | Security metrics are continuous values. DDQN uses neural networks to handle this. |
| **Off-policy learning** | Can learn from past experiences (replay buffer). Important because simulations are expensive. |
| **Reduced overestimation** | DDQN's dual-network approach gives more stable learning than vanilla DQN. |
| **Relatively simple** | Easy to implement and debug compared to more advanced methods. |

### More Modern / Potentially Better Alternatives

| Algorithm | What It Does Better | Trade-off |
|-----------|--------------------|-----------|
| **Dueling DQN** | Separates state-value from action-advantage estimation. Better when some states are bad regardless of action (e.g., nearly-compromised network). | Minor additional complexity |
| **Prioritised Experience Replay** | Replays important transitions more often (rare but high-reward events). Currently uses uniform random sampling. | More memory overhead |
| **PPO (Proximal Policy Optimisation)** | State-of-the-art policy gradient method. Better exploration, more stable training. Handles continuous actions too. | More complex, needs more tuning |
| **SAC (Soft Actor-Critic)** | Maximises entropy alongside reward — naturally explores more. State-of-the-art for continuous control. | Designed for continuous actions |
| **Rainbow DQN** | Combines 6 DQN improvements (DDQN + Dueling + PER + Noisy Nets + C51 + n-step). | Significantly more complex |
| **Multi-Agent RL (MARL)** | Model attacker AND defender as learning agents. More realistic adversarial dynamic. | Much harder to train |
| **Model-Based RL (e.g., Dreamer, MuZero)** | Learns a model of the environment and plans ahead. Could be more sample-efficient. | Complex, needs good world model |

### Most Promising Upgrade Path

For this specific problem, the most impactful improvements would likely be:

1. **Dueling DQN** — simple upgrade, separates "how good is this state" from "how much better is this action than others"
2. **Prioritised Experience Replay** — ensures the agent learns more from rare but important transitions (like near-compromise events)
3. **PPO** — if moving to a policy gradient approach, PPO is the industry standard for discrete action spaces

In [ ]:
# Comparison: DQN vs DDQN vs Dueling DQN (conceptual)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# DQN
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Regular DQN', fontsize=13, fontweight='bold', color='red')

ax.add_patch(mpatches.FancyBboxPatch((1, 7), 8, 2, boxstyle="round,pad=0.2", facecolor='#FFCDD2', edgecolor='#C62828', lw=2))
ax.text(5, 8.0, 'ONE Network', ha='center', fontsize=12, fontweight='bold')
ax.text(5, 7.4, 'Selects AND evaluates', ha='center', fontsize=10)

ax.add_patch(mpatches.FancyBboxPatch((1, 3), 8, 3, boxstyle="round,pad=0.2", facecolor='#FFEBEE', edgecolor='gray', lw=1))
ax.text(5, 5.3, 'Problem:', ha='center', fontsize=10, fontweight='bold', color='red')
ax.text(5, 4.6, 'Same network picks the\nbest action AND judges\nhow good it is', ha='center', fontsize=9)
ax.text(5, 3.5, '→ Overestimates Q-values', ha='center', fontsize=10, fontweight='bold', color='red')

ax.annotate('', xy=(5, 7), xytext=(5, 6.1), arrowprops=dict(arrowstyle='->', lw=2))

# DDQN (Joo Kai's approach)
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Double DQN (Used by Joo Kai)', fontsize=13, fontweight='bold', color='green')

ax.add_patch(mpatches.FancyBboxPatch((0.5, 7), 4, 2, boxstyle="round,pad=0.2", facecolor='#C8E6C9', edgecolor='#2E7D32', lw=2))
ax.text(2.5, 8.0, 'Main Net', ha='center', fontsize=11, fontweight='bold')
ax.text(2.5, 7.4, 'Selects action', ha='center', fontsize=10)

ax.add_patch(mpatches.FancyBboxPatch((5.5, 7), 4, 2, boxstyle="round,pad=0.2", facecolor='#BBDEFB', edgecolor='#1565C0', lw=2))
ax.text(7.5, 8.0, 'Target Net', ha='center', fontsize=11, fontweight='bold')
ax.text(7.5, 7.4, 'Evaluates value', ha='center', fontsize=10)

ax.add_patch(mpatches.FancyBboxPatch((1, 3), 8, 3, boxstyle="round,pad=0.2", facecolor='#E8F5E9', edgecolor='gray', lw=1))
ax.text(5, 5.3, 'Solution:', ha='center', fontsize=10, fontweight='bold', color='green')
ax.text(5, 4.6, 'Decouples selection from\nevaluation — reduces bias', ha='center', fontsize=9)
ax.text(5, 3.5, '→ More accurate Q-values', ha='center', fontsize=10, fontweight='bold', color='green')

ax.annotate('', xy=(5, 7), xytext=(3, 6.1), arrowprops=dict(arrowstyle='->', lw=1.5, color='#2E7D32'))
ax.annotate('', xy=(5, 7), xytext=(7, 6.1), arrowprops=dict(arrowstyle='->', lw=1.5, color='#1565C0'))

# Dueling DQN (potential upgrade)
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Dueling DQN (Potential Upgrade)', fontsize=13, fontweight='bold', color='#6A1B9A')

ax.add_patch(mpatches.FancyBboxPatch((0.5, 7), 4, 2, boxstyle="round,pad=0.2", facecolor='#E1BEE7', edgecolor='#6A1B9A', lw=2))
ax.text(2.5, 8.0, 'V(s) Stream', ha='center', fontsize=11, fontweight='bold')
ax.text(2.5, 7.4, '"How good is\nthis state?"', ha='center', fontsize=9)

ax.add_patch(mpatches.FancyBboxPatch((5.5, 7), 4, 2, boxstyle="round,pad=0.2", facecolor='#F3E5F5', edgecolor='#6A1B9A', lw=2))
ax.text(7.5, 8.0, 'A(s,a) Stream', ha='center', fontsize=11, fontweight='bold')
ax.text(7.5, 7.4, '"How much better\nis this action?"', ha='center', fontsize=9)

ax.add_patch(mpatches.FancyBboxPatch((1, 3), 8, 3, boxstyle="round,pad=0.2", facecolor='#F3E5F5', edgecolor='gray', lw=1))
ax.text(5, 5.3, 'Advantage:', ha='center', fontsize=10, fontweight='bold', color='#6A1B9A')
ax.text(5, 4.6, 'Knows some states are bad\nregardless of action', ha='center', fontsize=9)
ax.text(5, 3.5, '→ Q(s,a) = V(s) + A(s,a)', ha='center', fontsize=10, fontweight='bold', color='#6A1B9A')

ax.annotate('', xy=(5, 7), xytext=(3, 6.1), arrowprops=dict(arrowstyle='->', lw=1.5, color='#6A1B9A'))
ax.annotate('', xy=(5, 7), xytext=(7, 6.1), arrowprops=dict(arrowstyle='->', lw=1.5, color='#6A1B9A'))

plt.tight_layout()
plt.show()

## 12. Benchmarking: Is This Validated?

### How Joo Kai Benchmarked

Joo Kai's evaluation compares **against the simulator's own no-MTD baseline**:

1. Run the simulation with NO MTD → get baseline scores for 5 metrics (ASR, MTTC, APE, ROA, Risk)
2. Run the simulation WITH the DDQN agent → get scores for the same 5 metrics
3. Normalise: `score = metric_with_MTD / metric_without_MTD`
4. Sum the normalised scores to get an aggregate performance number

### What's Missing

| Gap | Why It Matters |
|-----|---------------|
| **No comparison to other scheduling strategies** in the paper | Should compare to: random MTD, round-robin, simultaneous deployment, fixed-interval (the simulator supports all of these!) |
| **No comparison to other RL algorithms** | Should compare to: vanilla DQN, PPO, or even simple heuristics |
| **No real-world validation** | Results only hold within the simulator's assumptions |
| **No cross-validation** | Results may be specific to the random seeds used |
| **Internal benchmark only** | No industry-standard cybersecurity benchmarks used |

### Industry Standard Benchmarks for Cyber RL

| Benchmark | What It Is | Relevance |
|-----------|-----------|----------|
| **CyberBattleSim** (Microsoft) | RL environment for enterprise network attack simulation | Direct competitor — models network lateral movement with RL agents |
| **CybORG** (CAGE Challenge) | Standardised cyber operations RL environment used in DARPA/CAGE competitions | Industry standard for RL-based cyber defence research |
| **NetworkAttackSimulator (NASim)** | Lightweight network attack simulation for RL research | Closest to MTDSim's approach — good comparison target |
| **MITRE ATT&CK Evaluations** | Real-world adversary emulation frameworks | Not RL-specific but provides realistic attack scenarios |
| **OpenAI Gym / Gymnasium** | Standard RL environment interface | MTDSim could be wrapped as a Gym environment for standardised comparison |

### For Your Work: Benchmarking Strategy

Your work should benchmark against:

1. **Joo Kai's DDQN model** (direct comparison, same simulator)
2. **The existing scheduling strategies** in the simulator (random, simultaneous, alternative)
3. **No-MTD baseline** (lower bound)
4. **Ideally**: Wrap the simulation as a Gym environment and compare to standard RL baselines

## 13. Critical Assessment: What Let Joo Kai Down?

### The Introduction and Lit Review Were Strong

Joo Kai told a clear story:
- Static defences fail → MTD is the answer → But existing MTD research is narrowly focused → RL can bridge the gap
- He correctly identified that temporal awareness was missing from prior work (Kreischer)
- He correctly identified scalability issues in EVADE
- He correctly identified the need for multi-strategy MTD rather than single-technique optimisation

### Where the Method/Execution Has Weaknesses

| Issue | Detail |
|-------|--------|
| **LSTM misuse** | Time-series input is shape (6,1) — one timestep. LSTMs need sequences to be useful. This is essentially a dense layer with extra overhead. Should use a sliding window of past N observations. |
| **`done` flag never set to True** | In `mtd_ai_training.py:184`, transitions are always stored with `done=False`. The agent never explicitly learns that compromise is a terminal failure. This is a significant bug. |
| **`host_compromise_ratio` has weight 0** | The most direct measure of security (how many hosts are compromised) doesn't influence the reward at all. |
| **No cost for doing nothing** | If all metrics stay stable, the reward is ~0 whether the agent deploys MTD or not. There's no penalty for inaction when the network is exposed but stable. |
| **No cost for deploying MTD** | No operational cost modelled. The agent can deploy MTD constantly without penalty, which isn't realistic. |
| **Normalisation instability** | Rewards are normalised against the replay buffer contents. Early in training, the buffer is small and non-representative, making early rewards unreliable. |
| **All weights are ±75 or 0** | No evidence of weight tuning. Why are APE and ASR equally important? The weight design seems arbitrary. |
| **Static degradation is a band-aid** | If no MTD deployed for >2000 time units, a random MTD is forced. This overrides the learned policy with a hard-coded rule. |
| **Evaluation sums normalised scores** | Summing normalised metrics assumes they're equally important and comparable after normalisation. No justification for this. |
| **100 episodes may not be enough** | RL typically requires thousands of episodes. With only 100 and train_start=1000, the agent may barely be trained. |

### Is the Simulation Sufficient?

**For answering "can RL select MTD strategies better than fixed schedules?"** — yes, the simulation is sufficient. It's a controlled environment to test the *concept*.

**For claiming real-world applicability** — no. The simulation abstracts away:
- Network performance impact of MTD
- Multi-attacker scenarios
- Adaptive attackers who learn the defender's patterns
- Partial observability (the agent sees perfect metrics)
- Time delays in metric collection

**The simulation is a tool** — it doesn't need to be perfectly realistic to answer research questions, but the claims need to be scoped to what the simulation can actually show. Joo Kai acknowledges this in Section 7 (Future Works) but the gap between claims and evidence is a weakness.

In [ ]:
# Visualise: Key bugs/issues in the implementation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Implementation Issues & Improvement Opportunities', fontsize=16, fontweight='bold', pad=20)

issues = [
    ('BUG', 'done=False always', 'Agent never learns that network\ncompromise is terminal failure', '#FFCDD2', '#C62828'),
    ('ISSUE', 'LSTM with 1 timestep', 'LSTM needs sequences; currently\nacts as expensive Dense layer', '#FFF9C4', '#F57F17'),
    ('ISSUE', 'HCR weight = 0', 'Most intuitive metric has no\ninfluence on reward', '#FFF9C4', '#F57F17'),
    ('DESIGN', 'No MTD cost modelled', 'Agent can deploy MTD endlessly\nwithout operational penalty', '#E3F2FD', '#1565C0'),
    ('DESIGN', 'No inaction penalty', 'Doing nothing when exposed has\nno explicit negative consequence', '#E3F2FD', '#1565C0'),
    ('DESIGN', 'All weights = ±75', 'No evidence of weight tuning;\nall active metrics equally important', '#E3F2FD', '#1565C0'),
    ('EVAL', 'Sum of normalised scores', 'Assumes metrics are equally\nimportant and comparable', '#E8F5E9', '#2E7D32'),
    ('EVAL', '100 episodes only', 'May be insufficient for\nRL convergence', '#E8F5E9', '#2E7D32'),
]

for i, (severity, title, desc, bg_color, text_color) in enumerate(issues):
    y = 9.0 - i * 1.1
    
    # Severity badge
    badge = mpatches.FancyBboxPatch((0.3, y - 0.35), 1.4, 0.7, boxstyle="round,pad=0.1",
                                     facecolor=bg_color, edgecolor=text_color, lw=1.5)
    ax.add_patch(badge)
    ax.text(1.0, y, severity, ha='center', va='center', fontsize=9, fontweight='bold', color=text_color)
    
    # Title and description
    ax.text(2.0, y + 0.1, title, fontsize=11, fontweight='bold', color=text_color)
    ax.text(6.5, y, desc, fontsize=9, va='center', family='monospace')

plt.tight_layout()
plt.show()

## 14. Summary: Key Takeaways

### What Joo Kai Built
A DDQN-based RL agent (MTDShield) that observes 11 network security metrics and decides which of 4 MTD strategies to deploy (or do nothing) to improve the network's security posture.

### The MDP
- **States**: 5 static features (APE, ASR, ROA, Risk, HCR) + 6 time-series features (MTTC, MTD frequency, time since last MTD, path variability, IP variability, attack type)
- **Actions**: Do nothing, IP Shuffle, OS Diversity, Service Diversity, Complete Topology Shuffle
- **Rewards**: Weighted sum of metric deltas (negative for bad metrics increasing, positive for good metrics increasing)
- **Transitions**: Determined by the SimPy simulation
- **Discount factor**: γ = 0.95

### The Architecture
Dual-pathway neural network: Dense layers for static features + LSTM for temporal features → concatenation → fusion → Q-value output.

### Key Findings
- Gamma 0.75-0.90 works best (balance between short and long-term thinking)
- Lower epsilon (0.5-0.7) outperformed higher values (early exploitation beats extensive exploration in short simulations)
- Higher train_start (2000) gave best results (more diverse experience before training)
- IDS sensitivity below 0.7 degrades performance sharply
- Both static and temporal modules contribute, but neither is individually critical (ablation showed only ~8% drop)

### What You Should Focus On for Your Work
1. Fix the `done` flag bug — terminal states should be marked
2. Consider giving `host_compromise_ratio` a non-zero weight
3. Use proper time-series windowing for LSTM (sliding window of last N states)
4. Add MTD deployment cost to the reward function
5. Compare against the simulator's built-in scheduling strategies (random, simultaneous, alternative)
6. Consider Dueling DQN or PPO as algorithm upgrades
7. Benchmark against CybORG or NASim if feasible

---

## Appendix A: Key File Locations

| Component | File |
|-----------|------|
| DDQN Network & Functions | `mtdnetwork/mtdai/mtd_ai.py` |
| Training Loop | `experiments/run.py` → `execute_ai_training()` |
| RL Training Operation (SimPy integration) | `mtdnetwork/operation/mtd_ai_training.py` |
| RL Inference Operation | `mtdnetwork/operation/mtd_ai_operation.py` |
| Training Script Example | `experiments/model_training/training_will.py` |
| Trained Models | `experiments/AI_model/` |
| Joo Kai's Thesis | `docs/thesis/JooKai/GENG5512Report_Tay_22489437.pdf` |

## Appendix B: Hyperparameter Reference

| Parameter | Default | What It Controls |
|-----------|---------|------------------|
| `gamma` | 0.95 | Discount factor — how much to value future vs immediate rewards |
| `epsilon` | 1.0 | Initial exploration rate (probability of random action) |
| `epsilon_min` | 0.01 | Minimum exploration rate |
| `epsilon_decay` | 0.995 | Per-episode decay multiplier for epsilon |
| `batch_size` | 32 | Number of experiences sampled per training step |
| `train_start` | 1000 | Minimum experiences before training begins |
| `episodes` | 100 | Number of full simulation runs for training |
| `memory maxlen` | 2000 | Maximum replay buffer size |
| `learning_rate` | 0.001 | Adam optimizer learning rate |
| `tau` | 0.1 | Soft update rate for target network (if used) |
| `static_degrade_factor` | 2000 | Force MTD deployment if idle this long |
| `mtd_interval` | 200 | Mean time between decision points (exponential) |
| `total_nodes` | 100 | Network size |
| `target_update_freq` | 10 episodes | How often target network gets hard-updated |